# Step 5: Cross-language test
Train on one language and test on another, for every pair. Same language: hold out one session at a time.
*Same as `src/05_cross_lingual.py`.*

### Dataset
**NoXi** (NOvice eXpert Interaction) - multimodal recordings of expert-novice conversations with continuous engagement labels.
- Dataset page: https://multimediate-challenge.org/datasets/Dataset_NoXi/
- Paper: A. Cafaro et al., "The NoXi Database: Multimodal Recordings of Mediated Novice-Expert Interactions", ACM ICMI 2017, https://doi.org/10.1145/3136755.3136780
- Used here: 9 conversations (English 027, 028, 030; French 001, 013, 021; German 070, 072, 077) from the NoXi train set:
  eGeMAPS voice features, OpenFace 2.0 face features, engagement labels (25 per second) and the pretrained-Transformer embeddings shipped with the corpus.
- Licence: NoXi is available for research under an End-User Licence Agreement (signed by the supervisor). The data is **not included** here and must not be shared;
  request access on the dataset page and place the downloaded zips as described in `01_extract_data`.

Go to the project folder.

In [ ]:
import os
if os.getcwd().endswith("notebooks"):
    os.chdir("..")          # work from the project folder

In [ ]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor

df = pd.read_parquet("data/features/noxi_engagement.parquet")
bookkeeping = ["face_0", "face_1", "face_2", "face_3", "face_4"]
cols = []
for c in df.columns:
    if (c.startswith("voice_") or c.startswith("face_")) and c not in bookkeeping:
        cols.append(c)
languages = ["English", "French", "German"]

In [ ]:
def ccc(y_true, y_pred):
    # Concordance Correlation Coefficient: 1 = perfect agreement
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    covariance = np.mean((y_true - y_true.mean()) * (y_pred - y_pred.mean()))
    return 2 * covariance / (y_true.var() + y_pred.var() + (y_true.mean() - y_pred.mean()) ** 2 + 1e-9)

**Train on one part, test on another, return the CCC.** First scale the features (fitted on the training part), then a Random Forest.

In [ ]:
def train_and_test(train, test):
    scaler = StandardScaler()
    X_train = scaler.fit_transform(train[cols].fillna(0))
    X_test = scaler.transform(test[cols].fillna(0))
    model = RandomForestRegressor(n_estimators=120, n_jobs=-1, random_state=42)
    model.fit(X_train, train["engagement"])
    return ccc(test["engagement"], model.predict(X_test))

**Every pair** (rows = train language, columns = test language).

In [ ]:
result = pd.DataFrame(index=languages, columns=languages, dtype=float)

for a in languages:
    for b in languages:
        if a == b:
            # same language: hold out one session at a time and average
            data = df[df["language"] == a]
            scores = []
            for s in data["session"].unique():
                scores.append(train_and_test(data[data["session"] != s], data[data["session"] == s]))
            result.loc[a, b] = np.mean(scores)
        else:
            # different language: train on all of a, test on all of b
            result.loc[a, b] = train_and_test(df[df["language"] == a], df[df["language"] == b])
        print(a, "->", b, round(result.loc[a, b], 3))

**Within vs cross language** and comparison with the saved result.

In [ ]:
within = np.mean([result.loc[l, l] for l in languages])
cross = np.mean([result.loc[a, b] for a in languages for b in languages if a != b])
print("within:", round(within, 3), " cross:", round(cross, 3), " drop:", round(100 * (within - cross) / within), "%")

saved = pd.read_csv("reports/noxi_crosslingual_ccc.csv", index_col=0)
print("same as saved:", np.allclose(saved.values, result.values))